In [ ]:
"""
Enhanced InsightFace Security Guard System - Optimized Version

This module provides an advanced security system using:
- YOLO for person detection, tracking, and pose estimation
- InsightFace for face detection, recognition, and attributes
- Face tracking, liveness detection, quality assessment
- Face attributes: age, gender, emotion, pose

Features:
1. Person Detection via YOLO (including pose keypoints)
2. Face Detection via InsightFace RetinaFace
3. Face Recognition (known vs unknown)
4. Face Attributes (age, gender, emotion, pose)
5. Face Tracking across frames
6. Liveness Detection
7. Quality Assessment
8. Pose Detection Integration

Installation:
    pip install insightface onnxruntime opencv-python numpy
"""

import cv2
import numpy as np
import pygame
import os
import time
import json
from pathlib import Path
from collections import OrderedDict, defaultdict
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field, asdict
import threading

# Import InsightFace
from insightface.app import FaceAnalysis
from ultralytics import solutions
from ultralytics.solutions.solutions import SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors
from ultralytics.utils import LOGGER
import asyncio
from threading import Lock


# ========== 📁 BASE DIRECTORY HELPER ==========
def _get_base_dir() -> Path:
    """Get the base directory of the project, handling both script and notebook modes."""
    try:
        return Path(__file__).parent.parent
    except NameError:
        return Path.cwd()



# ========== 🔊 SOUND SETUP ==========
BASE_DIR = _get_base_dir()
ALARM_FILE = BASE_DIR / "media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"

pygame.mixer.init()
_alarm_loaded = False

try:
    if ALARM_FILE.exists():
        pygame.mixer.music.load(str(ALARM_FILE))
        _alarm_loaded = True
        LOGGER.info(f"Alarm sound loaded: {ALARM_FILE}")
except Exception as e:
    LOGGER.warning(f"Failed to load alarm: {e}")


# ========== 📋 EVENT LOGGING ==========
class EventLogger:
    """Logs security events to file."""

    def __init__(self, log_file: str = None):
        base_dir = _get_base_dir()
        self.log_file = log_file or str(base_dir / "security_events.log")
        self._lock = Lock()

    def log(self, event_type: str, data: Dict):
        with self._lock:
            try:
                timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                log_entry = {"timestamp": timestamp, "event": event_type, **data}
                with open(self.log_file, "a") as f:
                    f.write(json.dumps(log_entry) + "\n")
            except Exception as e:
                print(f"[ERROR] Logging failed: {e}")


# ========== 📸 FACE SCREENSHOT CAPTURER ==========
class FaceScreenshotCapturer:
    """Captures and saves face screenshots with quality assessment."""

    def __init__(self, output_dir: str = "../captured_faces", min_quality: float = 30.0):
        self.output_dir = output_dir
        self.min_quality = min_quality

        self.known_dir = os.path.join(output_dir, "known")
        self.unknown_dir = os.path.join(output_dir, "unknown")

        for d in [self.known_dir, self.unknown_dir]:
            os.makedirs(d, exist_ok=True)

        self.best_faces: Dict[int, Dict] = {}
        self._unknown_count = 0
        self._lock = Lock()

        print(f"[INFO] Screenshots: {output_dir}")

    def assess_quality(self, frame: np.ndarray, box: List[float]) -> float:
        """Assess face quality: size, brightness, sharpness, contrast."""
        try:
            x1, y1, x2, y2 = map(int, box)
            h, w = frame.shape[:2]
            x1, x2 = max(0, x1), min(w, x2)
            y1, y2 = max(0, y1), min(h, y2)

            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                return 0.0

            # Size score
            size_score = min(100, ((x2 - x1) * (y2 - y1) / (w * h)) * 1000)

            # Brightness
            gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            brightness = np.mean(gray)
            bright_score = 100 - abs(brightness - 128) * 0.8

            # Sharpness
            lap = cv2.Laplacian(gray, cv2.CV_64F)
            sharp_score = min(100, lap.var() / 10)

            # Contrast
            contrast_score = min(100, np.std(gray) / 30 * 100)

            return size_score * 0.3 + bright_score * 0.25 + sharp_score * 0.25 + contrast_score * 0.2
        except Exception:
            return 0.0

    def update_best_face(
        self, track_id: int, box: List, frame: np.ndarray, is_known: bool, name: str = "Unknown"
    ) -> None:
        quality = self.assess_quality(frame, box)

        if quality < self.min_quality:
            return

        with self._lock:
            if track_id not in self.best_faces or quality > self.best_faces[track_id]["quality"]:
                self.best_faces[track_id] = {
                    "box": box.copy(),
                    "quality": quality,
                    "frame": frame.copy(),
                    "is_known": is_known,
                    "name": name,
                }

    def save_all(self) -> None:
        with self._lock:
            for tid, data in self.best_faces.items():
                try:
                    x1, y1, x2, y2 = map(int, data["box"])
                    h, w = data["frame"].shape[:2]
                    x1, x2 = max(0, x1), min(w, x2)
                    y1, y2 = max(0, y1), min(h, y2)

                    face = data["frame"][y1:y2, x1:x2]
                    if face.size == 0:
                        continue

                    if data["is_known"]:
                        path = os.path.join(
                            self.known_dir, f"{data['name']}_{data['quality']:.0f}_{int(time.time())}.jpg"
                        )
                    else:
                        self._unknown_count += 1
                        path = os.path.join(
                            self.unknown_dir,
                            f"unknown_{self._unknown_count}_{data['quality']:.0f}_{int(time.time())}.jpg",
                        )

                    cv2.imwrite(path, face)
                    print(f"[📸] Saved: {path}")
                except Exception as e:
                    print(f"[ERROR] Save failed: {e}")


# ========== 👤 FACE ATTRIBUTE ANALYZER ==========
class FaceAttributeAnalyzer:
    """Analyzes and formats face attributes."""

    EMOTION = {0: "😊", 1: "😐", 2: "😢", 3: "😠", 4: "😲", 5: "😨", 6: "😞"}
    GENDER = {0: "♂", 1: "♀"}

    @staticmethod
    def format(age: float, gender: float, emotion: np.ndarray, pose: Optional[np.ndarray] = None) -> str:
        age_str = f"{int(age)}y"
        gender_str = FaceAttributeAnalyzer.GENDER.get(int(gender > 0.5), "?")

        emo_idx = int(np.argmax(emotion)) if emotion is not None and len(emotion) > 0 else 1
        emo_str = FaceAttributeAnalyzer.EMOTION.get(emo_idx, "😐")

        # Pose info if available
        pose_str = ""
        if pose is not None and len(pose) >= 3:
            pitch, yaw, roll = pose[0], pose[1], pose[2]
            pose_str = f" | P:{int(pitch)}° Y:{int(yaw)}°"

        return f"{age_str} {gender_str} {emo_str}{pose_str}"


# ========== ⚙️ CONFIGURATION ==========
@dataclass
class SecurityGuardConfig:
    frame_interval: int = 10
    face_recognition_interval: float = 0.0
    enable_new_person_detection: bool = True
    person_cache_ttl: float = 30.0
    face_tolerance: float = 0.5
    min_face_size: Tuple[int, int] = (40, 40)
    insightface_model: str = "buffalo_l"
    insightface_det_size: Tuple[int, int] = (640, 640)
    enable_face_tracking: bool = True
    face_track_max_age: int = 30
    face_track_iou_threshold: float = 0.3
    enable_liveness: bool = True
    show_attributes: bool = True
    capture_faces: bool = True
    screenshot_dir: str = None
    min_face_quality: float = 30.0
    enable_logging: bool = True
    enable_keypoints_extraction: bool = False
    enable_keypoints_display: bool = False

    def validate(self) -> bool:
        """Validate configuration parameters."""
        if self.frame_interval < 0:
            raise ValueError("frame_interval must be non-negative")
        if not 0.3 <= self.face_tolerance <= 0.7:
            LOGGER.warning("face_tolerance should be between 0.3 and 0.7")
        if self.person_cache_ttl <= 0:
            raise ValueError("person_cache_ttl must be positive")
        if self.min_face_size[0] <= 0 or self.min_face_size[1] <= 0:
            raise ValueError("min_face_size must have positive dimensions")
        return True


# ========== 🔄 TRACKERS & CACHES ==========
class FaceTracker:
    """IoU-based face tracking."""

    def __init__(self, max_age: int = 30, iou_thresh: float = 0.3):
        self.max_age, self.iou_thresh = max_age, iou_thresh
        self.tracks, self.next_id = {}, 0
        self._lock = Lock()

    def _iou(self, b1: List[float], b2: List[float]) -> float:
        x1 = max(b1[0], b2[0])
        y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2])
        y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        union = (b1[2] - b1[0]) * (b1[3] - b1[1]) + (b2[2] - b2[0]) * (b2[3] - b2[1]) - inter
        return inter / union if union > 0 else 0.0

    def update(self, boxes, embeds, attrs) -> Dict:
        with self._lock:
            for t in self.tracks:
                self.tracks[t]["age"] += 1
            self.tracks = {k: v for k, v in self.tracks.items() if v["age"] < self.max_age}

            matched = set()
            for i, (box, emb, attr) in enumerate(zip(boxes, embeds, attrs)):
                best_iou, best_id = 0, None
                for tid, trk in self.tracks.items():
                    if tid in matched:
                        continue
                    iou = self._iou(box, trk["box"])
                    if iou > best_iou and iou >= self.iou_thresh:
                        best_iou, best_id = iou, tid

                if best_id:
                    self.tracks[best_id] = {"box": box, "embedding": emb, "attributes": attr, "age": 0}
                    matched.add(best_id)
                else:
                    self.tracks[self.next_id] = {"box": box, "embedding": emb, "attributes": attr, "age": 0}
                    matched.add(self.next_id)
                    self.next_id += 1

            return {t: inf for t, inf in self.tracks.items() if t in matched}


class PersonCache:
    def __init__(self, config: SecurityGuardConfig):
        self.config = config
        self._cache: OrderedDict = OrderedDict()
        self._lock = Lock()

    def get(self, pid: int) -> Optional[Tuple]:
        with self._lock:
            if pid in self._cache:
                n, isk, ts, attr = self._cache[pid]
                if time.time() - ts < self.config.person_cache_ttl:
                    self._cache.move_to_end(pid)
                    return (n, isk, attr)
                del self._cache[pid]
            return None

    def set(self, pid: int, name: str, is_known: bool, attrs: Dict = None) -> None:
        with self._lock:
            self._cache[pid] = (name, is_known, time.time(), attrs or {})


class NewPersonTracker:
    def __init__(self):
        self.seen, self.new = set(), set()
        self._lock = Lock()

    def update(self, ids: List[int]) -> List[int]:
        cur = set(ids)
        with self._lock:
            new_ids = cur - self.seen
            self.seen.update(cur)
            self.new.update(new_ids)
            return list(new_ids)

    def reset(self, pid: int) -> None:
        with self._lock:
            self.new.discard(pid)


class FrameController:
    def __init__(self, cfg: SecurityGuardConfig):
        self.cfg = cfg
        self.frames, self.last_time = 0, 0.0
        self._lock = Lock()

    def should_run(self, new_ids: List[int]) -> bool:
        with self._lock:
            self.frames += 1
            now = time.time()

            if self.cfg.enable_new_person_detection and new_ids:
                return True
            if self.cfg.face_recognition_interval > 0:
                if now - self.last_time >= self.cfg.face_recognition_interval:
                    self.last_time = now
                    return True
                return False
            if self.cfg.frame_interval > 0:
                return self.frames % self.cfg.frame_interval == 0
            return True


# ========== 🔍 INSIGHTFACE DETECTOR ==========
class InsightFaceDetector:
    def __init__(self, model: str = "buffalo_l", size: Tuple[int, int] = (640, 640)):
        self.model, self.size = model, size
        self.app = FaceAnalysis(name=model, providers=["CPUExecutionProvider"])
        self.app.prepare(ctx_id=0, det_size=size)
        print(f"[INFO] InsightFace: {model}")

    def detect(self, frame: np.ndarray) -> Tuple[List, List, List, List]:
        """Returns: boxes, embeddings, landmarks, attributes"""
        faces = self.app.get(frame)

        boxes, embeds, lands, attrs = [], [], [], []

        for face in faces:
            b = face.bbox
            if b[2] - b[0] < 40 or b[3] - b[1] < 40:
                continue

            boxes.append(b.tolist())
            embeds.append(face.embedding)
            lands.append(face.kps)
            attrs.append(
                {
                    "age": face.age,
                    "gender": face.gender,
                    "emotion": getattr(face, "emotion", np.array([0.5])),
                    "pose": getattr(face, "pose", np.array([0, 0, 0])),
                }
            )

        return boxes, embeds, lands, attrs


# ========== 🤖 MAIN SECURITY GUARD ==========
class EnhancedSecurityGuard(solutions.VisionEye):
    def __init__(
        self,
        *args,
        config: Optional[SecurityGuardConfig] = None,
        known_embs: List = None,
        known_names: List = None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.cfg = config or SecurityGuardConfig()
        self.cfg.validate()

        self.known_embs, self.known_names = known_embs or [], known_names or []
        self.sound_played = False

        # Components
        self.face_det = InsightFaceDetector(self.cfg.insightface_model, self.cfg.insightface_det_size)
        self.face_tracker = FaceTracker(self.cfg.face_track_max_age, self.cfg.face_track_iou_threshold)
        self.person_cache = PersonCache(self.cfg)
        self.new_tracker = NewPersonTracker()
        self.fr_controller = FrameController(self.cfg)
        self.event_log = EventLogger() if self.cfg.enable_logging else None

        if self.cfg.capture_faces:
            screenshot_dir = self.cfg.screenshot_dir
            if screenshot_dir is None:
                from pathlib import Path

                screenshot_dir = str(_get_base_dir() / "captured_faces")
            self.screenshot = FaceScreenshotCapturer(screenshot_dir, self.cfg.min_face_quality)
        else:
            self.screenshot = None

        # Stats
        self.stats = {
            "frames": 0,
            "fr_runs": 0,
            "cache_hits": 0,
            "new_triggers": 0,
            "faces_seen": 0,
            "known_detected": 0,
            "unknown_detected": 0,
            "poses_detected": 0,
        }

        print(f"[INFO] Known faces: {len(self.known_names)}")

    def play_alarm(self):
        if not self.sound_played:
            try:
                if not pygame.mixer.get_init():
                    pygame.mixer.init()
                if _alarm_loaded and not pygame.mixer.music.get_busy():
                    pygame.mixer.music.play()
                    self.sound_played = True
            except Exception:
                self.sound_played = True

    def stop_alarm(self):
        if self.sound_played:
            try:
                if _alarm_loaded:
                    pygame.mixer.music.stop()
            except Exception:
                pass
            self.sound_played = False

    def _identify(self, emb: np.ndarray) -> Tuple[str, bool]:
        if not self.known_embs:
            return "Unknown", False
        try:
            emb_arr = np.array(self.known_embs)
            q = emb.reshape(1, -1)
            qn = q / (np.linalg.norm(q) + 1e-5)
            en = emb_arr / (np.linalg.norm(emb_arr, axis=1, keepdims=True) + 1e-5)
            sims = np.dot(qn, en.T)
            best = np.argmax(sims)
            dist = 1 - sims[best]
            if dist < self.cfg.face_tolerance:
                return self.known_names[best], True
        except Exception:
            pass
        return "Unknown", False

    def _associate_face(self, face_box: List[float], person_boxes: List) -> Optional[int]:
        cx = (face_box[0] + face_box[2]) / 2
        cy = (face_box[1] + face_box[3]) / 2
        for i, pb in enumerate(person_boxes):
            if pb[0] <= cx <= pb[2] and pb[1] <= cy <= pb[3]:
                return i
        return None

    def __call__(self, im0: np.ndarray) -> SolutionResults:
        self.stats["frames"] += 1

        # YOLO extraction
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, self.line_width)

        # Get persons
        person_ids, person_boxes = [], []
        pose_keypoints = {}  # Store pose data per person

        # Extract YOLO pose keypoints if enabled
        if (
            self.cfg.enable_keypoints_extraction
            and hasattr(self.tracks, "keypoints")
            and self.tracks.keypoints is not None
        ):
            kpts_data = self.tracks.keypoints.xy.cpu().tolist() if hasattr(self.tracks.keypoints, "xy") else None
            if kpts_data:
                for i, (cls, tid) in enumerate(zip(self.clss, self.track_ids)):
                    if int(cls) == 0 and i < len(kpts_data):
                        pid = int(tid)
                        pose_keypoints[pid] = kpts_data[i]

        for cls, tid, box, conf in zip(self.clss, self.track_ids, self.boxes, self.confs):
            if int(cls) == 0:
                pid = int(tid)
                person_ids.append(pid)
                person_boxes.append(box.tolist())

        # Check new persons
        new_ids = self.new_tracker.update(person_ids)

        # FR decision
        run_fr = self.fr_controller.should_run(new_ids)

        face_boxes, face_embs, face_attrs = [], [], []

        if run_fr:
            self.stats["fr_runs"] += 1
            if new_ids:
                self.stats["new_triggers"] += len(new_ids)

            # InsightFace detection
            face_boxes, face_embs, _, face_attrs = self.face_det.detect(im0)
            self.stats["faces_seen"] += len(face_boxes)

            if self.cfg.enable_face_tracking:
                self.face_tracker.update(face_boxes, face_embs, face_attrs)

        unknown_count = 0
        detected_persons = []

        # Process each person
        for cls, tid, box, conf in zip(self.clss, self.track_ids, self.boxes, self.confs):
            if int(cls) != 0:
                annotator.box_label(box, label=self.adjust_box_label(cls, conf, tid), color=colors(int(tid), True))
                annotator.visioneye(box, self.vision_point)
                continue

            pid = int(tid)
            pbox = box.tolist()

            # Get from cache
            cached = self.person_cache.get(pid)

            if cached:
                name, is_known, attrs = cached
                self.stats["cache_hits"] += 1
            elif run_fr and face_embs:
                idx = self._associate_face([pbox[0], pbox[1], pbox[2], pbox[3]], face_boxes)

                if idx is not None and idx < len(face_embs):
                    name, is_known = self._identify(face_embs[idx])
                    attrs = face_attrs[idx] if idx < len(face_attrs) else {}

                    # Screenshot
                    if self.screenshot and idx < len(face_boxes):
                        self.screenshot.update_best_face(pid, face_boxes[idx], im0, is_known, name)

                    # Log pose data
                    if "pose" in attrs and attrs["pose"] is not None:
                        self.stats["poses_detected"] += 1
                else:
                    name, is_known, attrs = "Unknown", False, {}

                self.person_cache.set(pid, name, is_known, attrs)
            else:
                cached = self.person_cache.get(pid)
                if cached:
                    name, is_known, attrs = cached
                    self.stats["cache_hits"] += 1
                else:
                    name, is_known, attrs = "Unknown", False, {}

            if pid in new_ids:
                self.new_tracker.reset(pid)

            # Update counts
            if not is_known:
                unknown_count += 1
                self.stats["unknown_detected"] += 1
                color = (0, 0, 255)
            else:
                self.stats["known_detected"] += 1
                color = (0, 255, 0)

            # Build label
            label = f"{name}" if is_known else "Unknown"

            # Add attributes
            if self.cfg.show_attributes and attrs:
                attr_str = FaceAttributeAnalyzer.format(
                    attrs.get("age", 0),
                    attrs.get("gender", 0),
                    attrs.get("emotion", np.array([0.5])),
                    attrs.get("pose"),
                )
                label = f"{name}\n{attr_str}"

            # Final label
            base = self.adjust_box_label(int(cls), float(conf) if conf else 0.0, tid)
            prefix = str(self.CFG.get("person_label_prefix", label))
            final_label = f"{prefix}: {base}" if base else prefix

            # Draw
            annotator.box_label(box, label=final_label, color=colors(int(tid), True))
            annotator.visioneye(box, self.vision_point)

            # Draw YOLO pose keypoints if enabled
            if self.cfg.enable_keypoints_display and pid in pose_keypoints:
                kpts = pose_keypoints[pid]
                if kpts and len(kpts) > 0:
                    # Convert to numpy array for Annotator.kpts method
                    import numpy as np

                    if isinstance(kpts, list):
                        kpts_array = np.array(kpts, dtype=np.float32)
                    else:
                        kpts_array = kpts
                    annotator.kpts(kpts_array, shape=im0.shape[:2], kpt_line=True)

            # Store for logging
            detected_persons.append({"id": pid, "name": name, "is_known": is_known, "box": pbox, "attributes": attrs})

        # Alarm
        records = self.CFG.get("records", 1)
        if unknown_count >= records:
            self.play_alarm()
            if self.screenshot and run_fr:
                for fb in face_boxes:
                    self.screenshot.save_unknown_face(im0, fb)

            # Log event
            if self.event_log:
                self.event_log.log("ALARM", {"unknown_count": unknown_count, "persons": detected_persons})
        else:
            self.stop_alarm()

        # Output
        plot_im = annotator.result()
        self.display_output(plot_im)

        # Logging every 30 frames
        if self.stats["frames"] % 30 == 0:
            LOGGER.info(
                f"Frames: {self.stats['frames']} | FR: {self.stats['fr_runs']} | "
                f"Cache: {self.stats['cache_hits']} | Faces: {self.stats['faces_seen']} | "
                f"Poses: {self.stats['poses_detected']}"
            )

        # Overlay info
        cv2.putText(
            plot_im,
            f"Tracks: {len(person_ids)} | Known: {self.stats['known_detected']} | Unknown: {self.stats['unknown_detected']}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
        )

        status = f"FR: {'ON' if run_fr else 'OFF'} | Face: {len(face_boxes)} | Pose: {self.stats['poses_detected']}"
        if self.cfg.enable_keypoints_display:
            status += f" | Keypoints: {len(pose_keypoints)}"
        cv2.putText(plot_im, status, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))

    def save_screenshots(self):
        if self.screenshot:
            self.screenshot.save_all()


# ========== 🏭 FACTORY ==========
def create_security_guard(
    config: SecurityGuardConfig = None, face_dir: str = None, **kwargs
) -> EnhancedSecurityGuard:
    cfg = config or SecurityGuardConfig()

    det = InsightFaceDetector(cfg.insightface_model, cfg.insightface_det_size)

    embeds, names = [], []

    # Use default path if not provided
    if face_dir is None:
        face_dir = str(_get_base_dir() / "family_members")

    if os.path.exists(face_dir):
        for name in os.listdir(face_dir):
            pdir = os.path.join(face_dir, name)
            if not os.path.isdir(pdir):
                continue
            for f in os.listdir(pdir):
                try:
                    e = det.detect(cv2.imread(os.path.join(pdir, f)))[1]
                    if e:
                        embeds.append(e[0])
                        names.append(name)
                        LOGGER.info(f"Loaded: {name}/{f}")
                except Exception as e:
                    LOGGER.warning(f"Failed loading {name}/{f}: {e}")

    return EnhancedSecurityGuard(config=cfg, known_embs=embeds, known_names=names, **kwargs)


# ========== 🎬 MAIN ==========
if __name__ == "__main__":
    cfg = SecurityGuardConfig(
        frame_interval=10,
        enable_new_person_detection=True,
        person_cache_ttl=60.0,
        face_tolerance=0.5,
        enable_face_tracking=True,
        show_attributes=True,
        capture_faces=True,
        enable_logging=True,
    )

    print("\n" + "=" * 60)
    print("Enhanced Security Guard - YOLO + InsightFace + Pose")
    print("=" * 60 + "\n")

    cap = cv2.VideoCapture("../media_files/DSC_0098_edited.jpg")
    assert cap.isOpened(), "Video error"

    w, h, fps = (int(cap.get(x)) for x in (3, 4, 5))
    writer = cv2.VideoWriter("output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    guard = create_security_guard(
        config=cfg,
        show=True,
        model="yolo26n-pose.pt",  # Use pose model
        classes=[0, 2],
        vision_point=(w // 2 - 250, h - 10),
        conf=0.3,
        records=1,
    )

    while cap.isOpened():
        ok, im = cap.read()
        if not ok:
            break
        res = guard(im)
        writer.write(res.plot_im)

    print("\n[📸] Saving screenshots...")
    guard.save_screenshots()

    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    for k, v in guard.stats.items():
        print(f"  {k}: {v}")
    print("=" * 60)


In [ ]:

!yolo pose predict model=yolo26n-pose.pt source='../media_files/DSC_0098_edited.jpg' 

    Arguments received: ['yolo', 'pose', 'predict', 'model=yolo26n-pose.pt', "source='https://ultralytics.com/images/bus.jpg'", '#', 'predict', 'with', 'official', 'model']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['pose', 'obb', 'detect', 'segment', 'classify']
                MODE (required) is one of ['train', 'export', 'predict', 'track', 'benchmark', 'val']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo26n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predict model=yolo26n-seg.pt source='https://youtu.be/LNwODJXcvt4' imgsz=320

    3. Validate a pretrained detection model at batch-size 1 and image size 640:
        yolo val model=yolo26n.pt data=coco8.yaml batch=1 imgsz=640

    4. Export a YOLO26n classification model to ONNX format at image size 224 by 128 (no TASK required)
        yolo export model=yolo26n-cls.pt format=onnx imgsz=224,128

    5. Ultralytics solutions usage
        yolo solutions count or any of ['crop', 'blur', 'workout', 'heatmap', 'isegment', 'visioneye', 'speed', 'queue', 'analytics', 'inference', 'trackzone'] source="path/to/video.mp4"

    6. Run special commands:
        yolo help
        yolo checks
        yolo version
        yolo settings
        yolo copy-cfg
        yolo cfg
        yolo solutions help

    Docs: https://docs.ultralytics.com
    Solutions: https://docs.ultralytics.com/solutions/
    Community: https://community.ultralytics.com
    GitHub: https://github.com/ultralytics/ultralytics
